In [7]:
import pandas as pd

In [8]:
client_id = 'ab30884cbf2b41fda2e26aab4e3a7751'
client_secret = '06885b40fc684369a94080c5b1d87d45'

import spotipy
from spotipy.oauth2 import SpotifyClientCredentials

In [9]:

auth_manager = SpotifyClientCredentials(
    client_id=client_id,
    client_secret=client_secret
)

sp = spotipy.Spotify(auth_manager=auth_manager)

print("Connected to Spotify API")

Connected to Spotify API


In [10]:
# ── Playlist Mode Input ──
print('Paste your Spotify Playlist ID (the string after /playlist/ in the URL)')
print('Example: https://open.spotify.com/playlist/7zXhVAENGtOlG6Mnwk7bwv  →  7zXhVAENGtOlG6Mnwk7bwv')
playlist_id = input('Playlist ID: ').strip()
print(f'✅ Using playlist: {playlist_id}')


Paste your Spotify Playlist ID (the string after /playlist/ in the URL)
Example: https://open.spotify.com/playlist/7zXhVAENGtOlG6Mnwk7bwv  →  7zXhVAENGtOlG6Mnwk7bwv
✅ Using playlist: 1y1JSEK3NVTzdM5YSed53Q


In [11]:
results = sp.playlist_items("7zXhVAENGtOlG6Mnwk7bwv")

print(results)

HTTP Error for GET to https://api.spotify.com/v1/playlists/7zXhVAENGtOlG6Mnwk7bwv/items with Params: {'limit': 50, 'offset': 0, 'fields': None, 'market': None, 'additional_types': 'track,episode'} returned 401 due to Valid user authentication required


SpotifyException: http status: 401, code: -1 - https://api.spotify.com/v1/playlists/7zXhVAENGtOlG6Mnwk7bwv/items?limit=50&offset=0&additional_types=track%2Cepisode:
 Valid user authentication required, reason: None

In [ ]:
results = sp.playlist_items("7zXhVAENGtOlG6Mnwk7bwv")


In [ ]:
songs = []

for item in results["items"]:
    
    track = item.get("item")
    
    if track:
        
        artists = ", ".join([a["name"] for a in track["artists"]])
        
        songs.append({
            "track_name": track["name"],
            "album_name": track["album"]["name"],
            "artists": artists,
            "track_id": track["id"]
        })

playlist_df = pd.DataFrame(songs)

In [ ]:
playlist_df = pd.DataFrame(songs)

print(playlist_df)

                                   track_name  \
0                                       Lover   
1                                      Softly   
2                                     Excuses   
3                                     Pasoori   
4                                    With You   
5                                      Dil Nu   
6                           White Brown Black   
7                               Bijlee Bijlee   
8                             Suniyan Suniyan   
9                                    G.O.A.T.   
10                        Naina (From "Crew")   
11                                  Blue Eyes   
12                                  Love Dose   
13                              Desi Kalakaar   
14                                 Brown Rang   
15                                Millionaire   
16  Dil Chori (From "Sonu Ke Titu Ki Sweety")   
17                                  Old Money   
18                                   One Love   
19                  

In [ ]:


playlist_df.columns

Index(['track_name', 'album_name', 'artists', 'track_id'], dtype='object')

In [ ]:
df = pd.read_csv("dataset.csv")

In [ ]:
df["track_name"] = df["track_name"].str.lower()
df["album_name"] = df["album_name"].str.lower()
df["artists"] = df["artists"].str.lower()

playlist_df["track_name"] = playlist_df["track_name"].str.lower()
playlist_df["album_name"] = playlist_df["album_name"].str.lower()
playlist_df["artists"] = playlist_df["artists"].str.lower()

In [ ]:
df = df.drop_duplicates(subset=["track_id"])

In [ ]:
df["track_name"] = df["track_name"].str.lower()
playlist_df["track_name"] = playlist_df["track_name"].str.lower()

filtered_df = df[
    df.set_index(["track_name","album_name","artists"]).index.isin(
        playlist_df.set_index(["track_name","album_name","artists"]).index
    )
]

In [ ]:
print(len(filtered_df))


12


In [ ]:
print(len(songs))

50


In [ ]:
filtered_df = filtered_df.drop_duplicates(subset=["track_id"])

In [ ]:
filtered_df["track_name"].value_counts()

track_name
desi kalakaar                              1
love dose                                  1
blue eyes                                  1
dheere dheere                              1
brown rang                                 1
chaar botal vodka (from "ragini mms 2")    1
one bottle down                            1
lover                                      1
born to shine                              1
g.o.a.t.                                   1
na ja                                      1
bijlee bijlee                              1
Name: count, dtype: int64

In [ ]:
features = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo"
]

In [ ]:
playlist_features = filtered_df[features]

In [ ]:
playlist_vector = playlist_features.mean().values.reshape(1,-1)

In [ ]:
dataset_features = df[features]

In [ ]:


features = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo"
]

playlist_features = filtered_df[features]


dataset_features = df[features]



In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(playlist_features, dataset_features)

In [ ]:
import numpy as np

scores = similarity_matrix.mean(axis=0)

In [ ]:
dataset_without_playlist = df[~df["track_id"].isin(filtered_df["track_id"])]

In [ ]:
top_indices = np.argsort(scores)[::-1][:10]

recommended_songs = df.iloc[top_indices]

print(recommended_songs[["track_name","artists","album_name"]])

                                              track_name              artists  \
65650  all night (bts world original soundtrack) [pt. 3]       bts;juice wrld   
97555    se é pra falar de amor (tema novela a favorita)   mateus e cristiano   
35916                                  saudade né filha?           max e luan   
15339                                  forever right now       conor matthews   
1002                                             fellini               criolo   
30694                                          follow me   sam feldt;rita ora   
53853                        one more dance (with alida)          r3hab;alida   
83787                              perfect (feat. haris)  lucas & steve;haris   
83841                              perfect (feat. haris)  lucas & steve;haris   
99054                                           isabelle            zach hood   

                                              album_name  
65650  all night (bts world original soundtrack) 

---
## 🎵 Single Song Recommendation Mode
Instead of using a full playlist, you can recommend songs based on **one song**.
Just enter the song name (and optionally artist name) below.

In [ ]:
# ── Single Song Mode Input ──
print('Enter the song name you want recommendations for:')
SONG_NAME   = input('Song name : ').strip()
ARTIST_NAME = input('Artist name (optional, press Enter to skip): ').strip()

# Search Spotify for the song
query = f'track:{SONG_NAME}'
if ARTIST_NAME:
    query += f' artist:{ARTIST_NAME}'

search_results = sp.search(q=query, type='track', limit=5)
tracks = search_results['tracks']['items']

if not tracks:
    print('❌ Song not found. Try a different name.')
else:
    for i, t in enumerate(tracks):
        artists = ', '.join([a['name'] for a in t['artists']])
        print(f'[{i}] {t["name"]} — {artists}  (id: {t["id"]})')


Enter the song name you want recommendations for:
[0] Loved You First — One Direction  (id: 32HwMdkZuUmHg9uznhs9xM)


In [ ]:
# Pick the correct result index from the list above (default 0 = first result)
PICK_INDEX = 0

chosen_track = tracks[PICK_INDEX]
chosen_track_id = chosen_track["id"]
chosen_track_name = chosen_track["name"]
print(f"✅ Selected: {chosen_track_name} (id: {chosen_track_id})")

✅ Selected: Loved You First (id: 32HwMdkZuUmHg9uznhs9xM)


In [ ]:
# ── Match the chosen song in our dataset ──
features = [
    "danceability", "energy", "loudness", "speechiness",
    "acousticness", "instrumentalness", "liveness", "valence", "tempo"
]

matched = df[df["track_id"] == chosen_track_id]

if matched.empty:
    print("⚠️  Song not found in the local dataset by track_id.")
    print("   Trying by name...")
    matched = df[df["track_name"].str.lower() == chosen_track_name.lower()]

if matched.empty:
    print("❌ Song not in dataset. Try a different song.")
else:
    print(f"✅ Found in dataset: {matched.iloc[0]['track_name']} — {matched.iloc[0]['artists']}")

⚠️  Song not found in the local dataset by track_id.
   Trying by name...
❌ Song not in dataset. Try a different song.


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Use the first match
song_vector = matched.iloc[[0]][features].values  # shape (1, 9)

# Exclude the chosen song itself
other_songs = df[df["track_id"] != matched.iloc[0]["track_id"]].copy()

# Compute similarity
other_features = other_songs[features].values
sim_scores = cosine_similarity(song_vector, other_features)[0]   # shape (n,)

# Top 10
top_idx = np.argsort(sim_scores)[::-1][:10]
single_song_recs = other_songs.iloc[top_idx][["track_name", "artists", "album_name"]].reset_index(drop=True)

print(f"\n🎧 Top 10 songs similar to '{chosen_track_name}':\n")
print(single_song_recs.to_string(index=True))

IndexError: positional indexers are out-of-bounds